# gru_va board bring-up — fused-L20 configuration (`gru_va_fused_L20`)

**FUSED_L20_BRINGUP_V1 — 2026-08-07.** Derived from the P=40 bring-up notebook with three changes:

1. **fclk0 corrected to 100 MHz immediately after overlay load** — the hwh applies 62.5 MHz on *every* `Overlay()` call (root cause of the historical 427 k / "0.84 µs residual" numbers, superseded-with-annotation 2026-08-07). This cell must re-run after any bitstream reload.
2. Bitstream pair `gru_va_fused_L20.bit` / `.hwh` (provenance pair — basename-matched, never mix with `gru_va_P40`).
3. Kernel-of-record deltas for this artifact: **143 cyc/sample** (cosim-measured), fused output accumulation + mac3 sharing at L=20, **70 DSP** OOC. Golden expectations are the *certified digits* (fused csim is bit-identical to certified V2).

Chain state entering this notebook: csim ✓ → csynth ✓ → OOC impl ✓ (WNS +0.099, 70 DSP) → cosim ✓ (143 cyc) → overlay impl ✓ (record post-route WNS below before proceeding) → **board gate = this notebook**.

## 1. Load overlay + set fabric clock (one cell, inseparable)

In [ ]:
import time
import numpy as np
from pynq import Overlay, allocate
from pynq.ps import Clocks

BIT = "gru_L20_rodent_max_w20.bit"   # .hwh must sit beside it, basename-matched

ol = Overlay(BIT)
print("overlay loaded:", BIT)

# hwh re-applies 62.5 MHz on every load -- correct it HERE, every time.
print("fclk0 as loaded:", Clocks.fclk0_mhz, "MHz")
Clocks.fclk0_mhz = 100
assert abs(Clocks.fclk0_mhz - 100.0) < 0.5, f"fclk0 set failed: {Clocks.fclk0_mhz}"
print("fclk0 corrected :", Clocks.fclk0_mhz, "MHz  (design timing-closure point)")

## 2. Inspect IP names

In [ ]:
print(list(ol.ip_dict.keys()))

In [ ]:
# EDIT if the printed keys differ:
ip  = ol.gru_va_0
dma = ol.axi_dma_0
print(ip, "\n", dma)

## 3. Inspect the register map

In [ ]:
print(ip.register_map)

## 4. Helper functions (kernel start / done-poll)

In [ ]:
def start_kernel():
    ip.register_map.CTRL.AP_START = 1

def wait_done(timeout_s=10.0):
    t0 = time.time()
    while True:
        if int(ip.register_map.CTRL.AP_IDLE) == 1:
            return
        if time.time() - t0 > timeout_s:
            raise TimeoutError(f"kernel not idle after {timeout_s}s -- CTRL={ip.register_map.CTRL}")

print("helpers defined")

## 5. Control-path smoke test (mode=0 — a no-op in this kernel)

In [ ]:
rm = ip.register_map
rm.mode = 0
rm.n_samples = 0
rm.reset_state = 0
start_kernel()
wait_done()
print("control path OK (mode=0 no-op returned)")

## 6. GOLDEN RUN (mode=1) — the certification gate

Fused-L20 csim is **bit-identical** to certified V2, so the board must reproduce the **exact certified digits**:

- vs `golden_output.bin` (fixed-point reference): max |diff| = **0.0**
- vs float golden via `hls_metrics.py`: **cosine 1.000000 / ESR −73.70 dB / max 2.316236e-04 @ 2093 / RMSE 2.514294e-05 / mean −2.077e-05**

Any deviation from these digits is a red flag, not noise. Verify testdata hashes are the canonical `rodent_max_v2` set before scoring (weights 18CA9167… / input 607F8F6F… / output 0B2625DD…).

In [ ]:
x = np.fromfile("golden_input.bin", dtype=np.float32)
print(f"golden input: {x.size} samples")
assert x.size == 4096

ibuf = allocate(shape=(4096,), dtype=np.float32)
obuf = allocate(shape=(4096,), dtype=np.float32)
ibuf[:] = x
ibuf.flush()

In [ ]:
rm.mode = 1
rm.n_samples = 4096
rm.reset_state = 1

dma.recvchannel.transfer(obuf)
start_kernel()
dma.sendchannel.transfer(ibuf)

dma.sendchannel.wait()
dma.recvchannel.wait()
wait_done()

obuf.invalidate()
print("golden run complete")
print("first 5 outputs:", obuf[:5])

In [ ]:
g = np.fromfile("golden_output.bin", dtype=np.float32)
print("board[:5]: ", np.asarray(obuf[:5]))
print("golden[:5]:", g[:5])
d = np.max(np.abs(np.asarray(obuf) - g))
print(f"max |board - golden| over 4096 = {d:.3e}")
print("expect 0.0 exactly (bit-identical chain) -- any nonzero is a finding")

In [ ]:
obuf.tofile("board_golden_out_fused_L20.f32")
print("wrote board_golden_out_fused_L20.f32 -- verify byte size (16384) + today's date before copying")
print("score on PC:")
print("  python hls_metrics.py --hls board_golden_out_fused_L20.f32 --ref golden_output.bin --label board_golden_fused_L20")
print("PASS = cosine 1.000000 / ESR -73.70 / max 2.316236e-04 @ 2093 / RMSE 2.514294e-05")

## 7. Full-length chunked run (after the golden gate passes)

Pre-registration at fclk0 = 100 MHz, kernel 143 cyc/sample:
**ceiling 699 k samples/s; system is kernel-paced (2026-08-07 finding), so predict end-to-end 680–699 k = 14.2–14.6× RT.**
Rates materially below ~650 k → check `Clocks.fclk0_mhz` first (a reload silently resets it to 62.5).

In [ ]:
INFILE  = "anchor_nam_in.f32"
OUTFILE = "board_nam_out_fused_L20.f32"   # never clobber P40 captures
CHUNK   = 65536                            # locked for the campaign

x = np.fromfile(INFILE, dtype=np.float32)
n_total = x.size
print(f"{INFILE}: {n_total} samples")

ibuf_c = allocate(shape=(CHUNK,), dtype=np.float32)
obuf_c = allocate(shape=(CHUNK,), dtype=np.float32)
out = np.empty(n_total, dtype=np.float32)

In [ ]:
assert abs(Clocks.fclk0_mhz - 100.0) < 0.5, f"fclk0 is {Clocks.fclk0_mhz} -- re-run the load cell's clock fix"
t0 = time.time()
pos = 0
chunk_idx = 0
while pos < n_total:
    n = min(CHUNK, n_total - pos)

    ibuf_c[:n] = x[pos:pos+n]
    ibuf_c.flush()

    rm.mode = 1
    rm.n_samples = n
    rm.reset_state = 1 if chunk_idx == 0 else 0

    dma.recvchannel.transfer(obuf_c[:n] if n < CHUNK else obuf_c)
    start_kernel()
    dma.sendchannel.transfer(ibuf_c[:n] if n < CHUNK else ibuf_c)

    dma.sendchannel.wait()
    dma.recvchannel.wait()
    wait_done()

    obuf_c.invalidate()
    out[pos:pos+n] = obuf_c[:n]

    pos += n
    chunk_idx += 1
    if chunk_idx % 16 == 0:
        rate = pos / (time.time() - t0)
      #  print(f"  {pos}/{n_total}  ({rate/1e3:.0f} k samples/s)")

dt = time.time() - t0
print(f"done: {n_total} samples in {dt:.2f} s = {n_total/dt/1e3:.0f} k samples/s "
      f"({n_total/dt/48000:.1f}x real-time)")

out.tofile(OUTFILE)
print(f"wrote {OUTFILE} -- verify byte size ({4*n_total}) + date before scoring")

### Workbook numbers from this cell
- total samples / wall seconds / k samples/s / ×-real-time (printed above)
- OUTFILE name + INFILE provenance + bitstream basename (`gru_va_fused_L20`) + **fclk0 = 100 (recorded, not assumed)**
- score on the PC through the standard path: `hls_metrics.py --target ... --skip 1024 --window-sec 30`

## 8. Optional: single-chunk phase decomposition (sanity vs the P40 clock finding)

In [ ]:
# Expect sendW ~= 93-95 ms and total ~= 1.45-1.50 us/sample at 100 MHz.
n = CHUNK
ibuf_c[:n] = x[:n]; ibuf_c.flush()
rm.mode = 1; rm.n_samples = n; rm.reset_state = 0
tA = time.perf_counter()
dma.recvchannel.transfer(obuf_c)
start_kernel()
dma.sendchannel.transfer(ibuf_c)
dma.sendchannel.wait()
tH = time.perf_counter()
dma.recvchannel.wait()
wait_done()
obuf_c.invalidate()
tK = time.perf_counter()
print(f"sendW-dominated span {1e3*(tH-tA):.2f} ms, total {1e3*(tK-tA):.2f} ms = {(tK-tA)/n*1e6:.3f} us/sample")

In [ ]:
import time
from pynq import allocate
import numpy as np
assert abs(Clocks.fclk0_mhz - 100.0) < 0.5, f"fclk0 is {Clocks.fclk0_mhz}"
try:
    ibuf_c.freebuffer(); obuf_c.freebuffer()
except Exception:
    pass
ibuf_c = allocate(shape=(65536,), dtype=np.float32)
obuf_c = allocate(shape=(65536,), dtype=np.float32)
x = np.fromfile("anchor_nam_in.f32", dtype=np.float32)
CHUNK = 65536; REPS = 200
ibuf_c[:] = x[:CHUNK]; ibuf_c.flush()
rm.mode = 1; rm.n_samples = CHUNK; rm.reset_state = 1
dma.recvchannel.transfer(obuf_c); start_kernel(); dma.sendchannel.transfer(ibuf_c)
dma.sendchannel.wait(); dma.recvchannel.wait(); wait_done()
rm.reset_state = 0
hw = 0.0
t0 = time.time()
for _ in range(REPS):
    dma.recvchannel.transfer(obuf_c)
    start_kernel()
    dma.sendchannel.transfer(ibuf_c)
    ta = time.perf_counter()
    dma.sendchannel.wait()
    hw += time.perf_counter() - ta
    dma.recvchannel.wait()
    wait_done()
dt = time.time() - t0
print(f"total   : {1e6*dt/(REPS*CHUNK):.4f} us/sample = {100*dt/(REPS*CHUNK)*1e6:.2f} cyc")
print(f"hw-only : {1e6*hw/(REPS*CHUNK):.4f} us/sample = {100*hw/(REPS*CHUNK)*1e6:.2f} cyc  (sendW window)")

## 9. Cleanup (only when fully done — buffers are reusable across runs)

In [ ]:
ibuf.freebuffer(); obuf.freebuffer()
ibuf_c.freebuffer(); obuf_c.freebuffer()
print("buffers freed -- reallocate before any reuse (freed PynqBuffers fail only on access)")